# Web Search Tool with Anthropic

This notebook demonstrates how to configure and use the built-in web search tool with Anthropic.

It covers:
- Environment and model setup
- Conversation helper functions
- Web search tool configuration
- A practical search example

## Environment Setup

Load environment variables, initialize the Anthropic client, and choose the model for web-enabled responses.

In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper Functions

These helpers keep message history consistent and provide a single function for model calls with optional tools.

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Web Search Tool Configuration

Define the web search schema with usage limits and optional domain controls to keep retrieval focused.

In [ ]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

## Example Query

Use this cell to run a search-oriented prompt and inspect the model response.

In [ ]:
messages = []
add_user_message(
    messages,
    """
    What are evidence-based exercises for increasing leg muscle mass?
    """,
)
response = chat(messages, tools=[web_search_schema])
response